# MARBERT Fine-tuning: Joint Learning versus Sentiment Only

**Status: not executed in the onboarding CPU environment.** Hugging Face model downloads were denied by the environment network policy. This notebook is ready for a GPU runtime with model access; it is not evidence of completed MARBERT training.

The shared MARBERT encoder feeds two classification heads. Joint loss is sentiment cross-entropy + sarcasm cross-entropy. The control uses sentiment loss alone. Both select their best epoch using validation sentiment macro-F1.

Recommended: a GPU runtime; batch size 8 and sequence length 128 are starting settings, not guaranteed memory requirements. Reduce batch size if necessary. Run the following installation cell in a fresh notebook runtime.

In [1]:
from pathlib import Path
import os, subprocess

# Colab opens the notebook without checking out the rest of the repository.
if not Path("pyproject.toml").exists():
    if Path.cwd().name == "notebooks" and (Path.cwd().parent / "pyproject.toml").exists():
        os.chdir(Path.cwd().parent)
    else:
        checkout = Path("/content/NLP_Capstone") if Path("/content").exists() else Path.cwd() / "NLP_Capstone"
        if not checkout.exists():
            subprocess.run(["git", "clone", "https://github.com/iYm10/NLP_Capstone.git", str(checkout)], check=True)
        os.chdir(checkout)
assert Path("pyproject.toml").exists(), "Project checkout was not found."
print("Project directory:", Path.cwd())

Project directory: /content/NLP_Capstone


In [2]:
%pip install -e ".[train,dev]"

Obtaining file:///content/NLP_Capstone
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 100.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 21.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 117.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 125.8 MB/s  0:00:00
  Building editable for maqsad (pyproject.toml) ... done
  Created wheel for maqsad: filename=maqsad-0.1.0-0.editable-py3-none-any.whl size=1433 sha256=444026e1cc84443e6fdafa0d6a9b81de7f9f5751d84a8abebaa77b0862b6b228
  Stored in directory: /tmp/pip-ephem-wheel-cache-rl0ibvrz/wheels/ab/42/f3/3dc6e500742de9a3fcf3fa58055107f170b38d969931182c9f
Successfully built maqsad
  Attempting uninstall: huggingface-hub
    Found existing installation: huggin

In [3]:
from pathlib import Path
import os, subprocess, sys, json
ROOT = Path.cwd()
if ROOT.name == "notebooks": ROOT = ROOT.parent
os.chdir(ROOT)
import torch
print("GPU available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Use a GPU runtime for the complete MARBERT experiment."

GPU available: True


## Model revision
Resolve and record an immutable model revision before both runs. Internet access to Hugging Face and its artifact hosts is required. The dataset downloader separately verifies immutable data hashes.

## Persistent checkpoint storage
In Google Colab, authorize the Google Drive mount when prompted. Training writes directly to `MyDrive/Maqsad/artifacts`, so completed checkpoints survive runtime deletion. A new run uses a separate timestamped directory. Outside Colab, outputs remain in the local artifacts directory. A disconnected runtime does not automatically resume an interrupted epoch.

In [4]:
from datetime import datetime, timezone
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
try:
    from google.colab import drive
except ImportError:
    OUTPUT_ROOT = ROOT / "artifacts" / RUN_ID
else:
    drive.mount("/content/drive")
    OUTPUT_ROOT = Path("/content/drive/MyDrive/Maqsad/artifacts") / RUN_ID
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print("Checkpoints and reports will be saved to:", OUTPUT_ROOT)

Mounted at /content/drive
Checkpoints and reports will be saved to: /content/drive/MyDrive/Maqsad/artifacts/20261102T192207Z


In [5]:
from huggingface_hub import model_info
REVISION = model_info("UBC-NLP/MARBERT").sha
print("MARBERT revision:", REVISION)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


MARBERT revision: 88e1fa192dd723cf0b3563500aec46209762eb22


In [6]:
for task in ["sentiment", "joint"]:
    subprocess.run([sys.executable, "-m", "maqsad.train", "--task", task, "--revision", REVISION,
        "--epochs", "3", "--batch-size", "8", "--seed", "42", "--output", str(OUTPUT_ROOT / f"marbert-{task}")], check=True)

In [7]:
import pandas as pd
reports = {task: json.loads((OUTPUT_ROOT / f"marbert-{task}" / "metrics.json").read_text()) for task in ["sentiment", "joint"]}
display(pd.DataFrame({task: {"sentiment_macro_f1": report["test"]["sentiment_macro_f1"], "sentiment_accuracy": report["test"]["sentiment_accuracy"]} for task, report in reports.items()}))
print("Joint sarcasm F1:", reports["joint"]["test"]["sarcasm_f1"])

,sentiment,joint
sentiment_macro_f1,0.657118,0.655661
sentiment_accuracy,0.719000,0.700000


Joint sarcasm F1: 0.5855745721271394


In [10]:
import sys
import importlib
from pathlib import Path

project = Path("/content/NLP_Capstone")
assert (project / "src/maqsad").exists(), "Project folder is missing"

sys.path.insert(0, str(project / "src"))
importlib.invalidate_caches()

from maqsad.inference import predict_transformer

predict_transformer(
    ["يا سلام على الخدمة، ساعتين وأنا أنتظر!"],
    checkpoint=Path(OUTPUT_ROOT) / "marbert-joint",
)

[{'sentiment': 'negative',
  'sentiment_score': 0.4662593901157379,
  'sarcasm': False,
  'sarcasm_score': 0.005829364061355591}]

## Interpretation
A positive difference for a single seed is preliminary. Repeat both tasks with identical seeds 42, 43 and 44 and report mean and standard deviation. Do not tune on the test set. Copy only final aggregate metrics into reports; keep checkpoints and raw tweets out of Git.

## Export reports for GitHub
This small archive contains aggregate metrics and run metadata. It excludes large model weights, which remain in the checkpoint directory. Download the executed notebook separately using **File → Download → Download .ipynb**. The export does not push to GitHub or require a GitHub token.

In [11]:
import shutil, platform
export_dir = OUTPUT_ROOT / "github-results"
export_dir.mkdir(exist_ok=True)
for task in ["sentiment", "joint"]:
    shutil.copy2(OUTPUT_ROOT / f"marbert-{task}" / "metrics.json", export_dir / f"marbert-{task}-metrics.json")
(export_dir / "run.json").write_text(json.dumps({"run_id": RUN_ID, "model_revision": REVISION,
    "python": platform.python_version(), "torch": torch.__version__,
    "gpu": torch.cuda.get_device_name(0), "source_commit": subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()}, indent=2))
archive = shutil.make_archive(str(OUTPUT_ROOT / "maqsad-results"), "zip", export_dir)
print("Results archive:", archive)
try:
    from google.colab import files
except ImportError:
    pass
else:
    files.download(archive)

Results archive: /content/drive/MyDrive/Maqsad/artifacts/20261102T192207Z/maqsad-results.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>